# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad:** pesos constantes del ultimo mes disponible

**Alcance titulares:** Sector Publico Total = Adm. Nacional + PAMI + Fondos Fiduciarios  
**Comparacion del ajuste:** anual 2023 vs 2024 y 2023 vs 2025 (anos completos)

### Graficos generados
1. Resultado Primario y Financiero mensual 2020-2026
2. Composicion del gasto corriente (mensual)
3. Composicion de ingresos (mensual)
4. Transferencias a provincias (mensual)
5. Variacion anual por componente AIF (2023 vs 2024 y 2025)
6. Torta: % de la baja del gasto por rubro (2023 vs 2025, IMIG)
7. Barras: variacion real por rubro IMIG (2023 vs 2024 y 2025)

In [ ]:
# ── Celda 1: Dependencias y helpers ──────────────────────────────────
import sys, io, os, zipfile, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DPI      = 600
FIG_W    = 26
FIG_H    = 7
GRAFICOS = []
GAP_DATE = pd.Timestamp('2022-06-01')

ANO_BASE = 2023
ANO_1    = 2024
ANO_2    = 2025

# Nombres de columna ASCII (evita KeyError por encoding unicode)
COL_BASE = str(ANO_BASE)
COL_1    = str(ANO_1)
COL_2    = str(ANO_2)
COL_V1   = f'Var {ANO_BASE}->{ANO_1} (B)'
COL_V2   = f'Var {ANO_BASE}->{ANO_2} (B)'
COL_P1   = f'% {ANO_BASE}->{ANO_1}'
COL_P2   = f'% {ANO_BASE}->{ANO_2}'

plt.rcParams['figure.dpi']        = 100
plt.rcParams['savefig.dpi']       = DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 9

MESES_ES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
            7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

def drop_gap(serie):
    return serie.drop(GAP_DATE, errors='ignore')

def fmt_int(ax, idx):
    ticks, labels = [], []
    for i, d in enumerate(idx):
        if d.month in [1, 4, 7, 10]:
            ticks.append(i)
            labels.append(f"{MESES_ES[d.month]}-{d.year}")
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_xlim(-0.8, len(idx) - 0.2)

def save_fig(fig, nombre):
    fname = f'{nombre}.png'
    fig.savefig(fname, dpi=DPI, bbox_inches='tight')
    GRAFICOS.append(fname)
    print(f'  Guardado: {fname}')

def vline_milei(ax, idx):
    milei = pd.Timestamp('2023-12-01')
    if milei in list(idx):
        ax.axvline(list(idx).index(milei), color='#e67e22',
                   lw=1.5, ls='--', alpha=0.7, label='Inicio Milei (Dic-2023)')

print('OK')
print(f'Comparacion: {ANO_BASE} vs {ANO_1} y {ANO_BASE} vs {ANO_2}')

In [ ]:
# ── Celda 2: Carga de datos y funciones ──────────────────────────────
REPO     = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

df_aif      = pd.read_csv(f'{REPO}/output/aif_consolidado.csv', parse_dates=['fecha'])
df_imig_raw = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
df_imig     = df_imig_raw.drop_duplicates(subset=['fecha','concepto_codigo','nivel_jerarquia']).copy()
df          = df_aif[df_aif['periodo'] == 'mensual'].copy()

ipc_raw = pd.read_excel(
    io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
    usecols=['date', 'Nivel general'])
ipc_raw['date'] = pd.to_datetime(ipc_raw['date'])
ipc = ipc_raw.set_index('date')['Nivel general'].sort_index()
ipc.index = ipc.index.to_period('M').to_timestamp()

BASE      = ipc.index.max()
IPC_BASE  = ipc.loc[BASE]
MES_FINAL = df['fecha'].max()

def deflactar(serie):
    idx = serie.index.to_period('M').to_timestamp()
    return serie.values * (IPC_BASE / ipc.reindex(idx).values)

def get_serie(concepto, subsector='total_adm_nacional'):
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

def get_serie_total(concepto):
    s_nac  = get_serie(concepto, 'total_adm_nacional')
    s_pami = get_serie(concepto, 'pami_fdos_otros')
    s_gen  = get_serie(concepto, 'total_general')
    s_suma = s_nac.add(s_pami.reindex(s_nac.index).fillna(0))
    if len(s_gen) > 0:
        s_suma.update(s_gen)
    return s_suma.sort_index()

def get_real_anual(concepto, subsector='total_adm_nacional'):
    s = get_serie(concepto, subsector)
    r = pd.Series(deflactar(s), index=s.index)
    return r.groupby(r.index.year).sum() / 1e6

def get_real_anual_total(concepto):
    s = get_serie_total(concepto)
    r = pd.Series(deflactar(s), index=s.index)
    return r.groupby(r.index.year).sum() / 1e6

def get_imig_anual(concepto, nivel=None):
    mask = df_imig['concepto_codigo'] == concepto
    if nivel is not None:
        mask &= df_imig['nivel_jerarquia'] == nivel
    s = df_imig[mask].copy()
    s['ipc_v'] = ipc.reindex(s['fecha'].dt.to_period('M').dt.to_timestamp()).values
    s['real']  = s['valor_millones_pesos'] * (IPC_BASE / s['ipc_v'])
    return s.groupby(s['fecha'].dt.year)['real'].sum() / 1e6

print(f'AIF mensual  : {len(df):,} registros | {df.fecha.min().strftime("%Y-%m")} - {MES_FINAL.strftime("%Y-%m")}')
print(f'IMIG (dedup) : {len(df_imig):,} registros')
print(f'Base deflac. : {BASE.strftime("%Y-%m")} (IPC = {IPC_BASE:,.0f})')

In [ ]:
# ── Celda 3: Grafico 01 - Resultado Primario y Financiero ─────────────
primario   = drop_gap(get_serie_total('XIV_RESULTADO_PRIMARIO'))
financiero = drop_gap(get_serie_total('XV_RESULTADO_FINANCIERO'))
prim_real  = deflactar(primario)
fin_real   = deflactar(financiero)
idx = primario.index
pos = list(range(len(idx)))

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in prim_real]
ax.bar(pos, prim_real / 1e6, color=colors, alpha=0.9, zorder=2)
ax.plot([p + 0.5 for p in pos], fin_real / 1e6, 'o-', color='#2c3e50',
        lw=1.8, ms=4, zorder=3)
ax.axhline(0, color='black', lw=1.0)
vline_milei(ax, idx)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)

# Leyenda manual con verde (superavit), rojo (deficit) y linea financiero
legend_handles = [
    Patch(facecolor='#27ae60', alpha=0.9, label='Resultado Primario (superavit)'),
    Patch(facecolor='#c0392b', alpha=0.9, label='Resultado Primario (deficit)'),
    Line2D([0], [0], marker='o', color='#2c3e50', lw=1.8, ms=4, label='Resultado Financiero'),
    Line2D([0], [0], color='#e67e22', lw=1.5, ls='--', alpha=0.7, label='Inicio Milei (Dic-2023)'),
]
ax.legend(handles=legend_handles, fontsize=9)
plt.tight_layout()
save_fig(fig, '01_resultado_primario_financiero')
plt.show()

df_anual = pd.DataFrame(
    {'Primario real (B)': prim_real / 1e6, 'Financiero real (B)': fin_real / 1e6},
    index=idx
)
print(f'\nResumen anual - Sector Publico Total (real, {BASE.strftime("%b %Y")}):')
print(df_anual.groupby(df_anual.index.year).sum().round(1).to_string())

In [ ]:
# ── Celda 4: Grafico 02 - Composicion del gasto corriente ────────────
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Subsidios',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
}
colors_g = ['#c0392b','#8e44ad','#2980b9','#16a085','#6c3483','#d68910']
idx_full = get_serie('II_GASTOS_CORRIENTES').index
idx = idx_full[idx_full != GAP_DATE]
pos = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))
for (cod, label), color in zip(comp_gasto.items(), colors_g):
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(pos, vals, bottom=bottom, label=label, color=color, alpha=0.88, zorder=2)
    bottom += vals
vline_milei(ax, idx)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '02_composicion_gasto')
plt.show()

In [ ]:
# ── Celda 5: Grafico 03 - Composicion de ingresos ────────────────────
idx_full = get_serie('I_INGRESOS_CORRIENTES').index
idx = idx_full[idx_full != GAP_DATE]
pos = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))
for cod, label, color in [
    ('I1_INGRESOS_TRIBUTARIOS',    'Tributarios',    '#1e8449'),
    ('I2_APORTES_SEG_SOCIAL',      'Seg. Social',    '#1a5276'),
    ('I3_INGRESOS_NO_TRIBUTARIOS', 'No tributarios', '#b7950b'),
]:
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(pos, vals, bottom=bottom, label=label, color=color, alpha=0.88, zorder=2)
    bottom += vals
vline_milei(ax, idx)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '03_composicion_ingresos')
plt.show()

In [ ]:
# ── Celda 6: Grafico 04 - Transferencias a provincias ────────────────
corr_tes  = drop_gap(get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional'))
cap_tes   = drop_gap(get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional'))
corr_real = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real  = pd.Series(deflactar(cap_tes),  index=cap_tes.index)
idx = corr_real.index
pos = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
c_vals = corr_real.values / 1000
k_vals = cap_real.reindex(idx).fillna(0).values / 1000
ax.bar(pos, c_vals, color='#6c3483', alpha=0.9, label='Corrientes (Tesoro)', zorder=2)
ax.bar(pos, k_vals, bottom=c_vals, color='#ca6f1e', alpha=0.9, label='Capital (Tesoro)', zorder=2)
vline_milei(ax, idx)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '04_transferencias_provincias')
plt.show()

In [ ]:
# ── Celda 7: Grafico 05 - Variacion anual AIF 2023 vs 2024/2025 ───────
conceptos_aif = {
    'I_INGRESOS_CORRIENTES':        'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':         'Gastos corrientes',
    'II2_INTERESES':                '  Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':  '  Prestaciones Seg.Social',
    'II1a_REMUNERACIONES':          '  Remuneraciones',
    'II4b1_TRANSF_PROVINCIAS_CABA': '  Transf. Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':   '  Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':   '  Subsidios',
    'V_GASTOS_CAPITAL':             'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':       'RESULTADO PRIMARIO',
}

rows = []
for cod, nombre in conceptos_aif.items():
    anual = get_real_anual(cod)
    v23 = anual.get(ANO_BASE)
    v24 = anual.get(ANO_1)
    v25 = anual.get(ANO_2)
    if v23 is None: continue
    var24 = round(v24 - v23, 1) if v24 is not None else None
    var25 = round(v25 - v23, 1) if v25 is not None else None
    pct24 = f'{(v24/v23-1)*100:+.1f}%' if v24 is not None and v23 != 0 else None
    pct25 = f'{(v25/v23-1)*100:+.1f}%' if v25 is not None and v23 != 0 else None
    rows.append({
        'Concepto': nombre,
        COL_BASE:   round(v23, 1),
        COL_1:      round(v24, 1) if v24 is not None else None,
        COL_2:      round(v25, 1) if v25 is not None else None,
        COL_V1:     var24,
        COL_V2:     var25,
        COL_P1:     pct24,
        COL_P2:     pct25,
    })

df_aif_comp = pd.DataFrame(rows)
print(f'Variacion anual real: {ANO_BASE} vs {ANO_1}/{ANO_2} (AIF, Adm. Nacional)')
print(df_aif_comp.to_string(index=False))

mask_det = df_aif_comp['Concepto'].str.startswith('  ')
df_det   = df_aif_comp[mask_det].copy()
df_det['Concepto'] = df_det['Concepto'].str.strip()
df_det = df_det.sort_values(COL_V2)
y_pos  = np.arange(len(df_det))
h      = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
c24 = ['#c0392b' if v < 0 else '#1a5276' for v in df_det[COL_V1].fillna(0)]
c25 = ['#922b21' if v < 0 else '#154360' for v in df_det[COL_V2].fillna(0)]
ax.barh(y_pos + h/2, df_det[COL_V1].fillna(0), height=h, color=c24,
        alpha=0.85, edgecolor='white', linewidth=0.4)
ax.barh(y_pos - h/2, df_det[COL_V2].fillna(0), height=h, color=c25,
        alpha=0.85, edgecolor='white', linewidth=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_det['Concepto'], fontsize=9)
ax.axvline(0, color='#2c3e50', lw=1.2)
ax.set_xlabel(f'Variacion real anual | Billones ARS ({BASE.strftime("%b %Y")})')
ax.grid(axis='x', alpha=0.2)
ax.spines['left'].set_visible(False)
# Leyenda manual con todos los colores posibles
leg = [
    Patch(facecolor='#c0392b', alpha=0.85, label=f'{ANO_BASE}->{ANO_1} (baja)'),
    Patch(facecolor='#1a5276', alpha=0.85, label=f'{ANO_BASE}->{ANO_1} (suba)'),
    Patch(facecolor='#922b21', alpha=0.85, label=f'{ANO_BASE}->{ANO_2} (baja)'),
    Patch(facecolor='#154360', alpha=0.85, label=f'{ANO_BASE}->{ANO_2} (suba)'),
]
ax.legend(handles=leg, fontsize=8, loc='lower right')
plt.tight_layout()
save_fig(fig, '05_ajuste_componentes_anual')
plt.show()

In [ ]:
# ── Celda 8: Graficos 06 y 07 - Torta y barras por rubro IMIG ────────
rubros = [
    ('Obra publica',         'Gastos_capital',               1),
    ('Subsidios',            'Subsidios_economicos',          1),
    ('Transf. Provincias',   'Transf_corrientes_provincias',  1),
    ('Salarios',             'Salarios',                      2),
    ('Jubilaciones',         'Jubilaciones_pensiones',         2),
    ('Universidades',        'Transf_universidades',           1),
    ('Pensiones NC',         'Pensiones_no_contributivas',     2),
    ('PAMI',                 'INSSJP_PAMI',                    2),
    ('Otros prog. sociales', 'Otros_prog_sociales',            2),
    ('AUH',                  'AUH',                            2),
]

filas_imig = []
for nombre, cod, nivel in rubros:
    anual = get_imig_anual(cod, nivel)
    v23 = anual.get(ANO_BASE)
    v24 = anual.get(ANO_1)
    v25 = anual.get(ANO_2)
    if v23 is None or v25 is None:
        print(f'  Sin datos: {nombre}')
        continue
    var25 = round(v25 - v23, 1)
    var24 = round(v24 - v23, 1) if v24 is not None else None
    pct25 = f'{(v25/v23-1)*100:+.1f}%' if v23 != 0 else 'n/d'
    filas_imig.append({
        'Rubro':  nombre,
        COL_BASE: round(v23, 1),
        COL_1:    round(v24, 1) if v24 is not None else None,
        COL_2:    round(v25, 1),
        COL_V1:   var24,
        COL_V2:   var25,
        COL_P2:   pct25,
    })

df_ir = pd.DataFrame(filas_imig)
print(f'Variacion anual IMIG: {ANO_BASE} vs {ANO_1}/{ANO_2}')
print(df_ir.to_string(index=False))

# ── Grafico 06: Torta sin titulo principal ────────────────────────────
df_baja    = df_ir[df_ir[COL_V2] < 0].copy()
df_baja['abs_var'] = df_baja[COL_V2].abs()
total_baja = df_baja['abs_var'].sum()
colors_pie = ['#922b21','#1a5276','#6c3483','#1e8449','#784212','#0e6655','#4d5656']

fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    df_baja['abs_var'], labels=df_baja['Rubro'],
    autopct='%1.1f%%', colors=colors_pie[:len(df_baja)],
    startangle=140, pctdistance=0.75, labeldistance=1.12,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2}
)
for at in autotexts:
    at.set_fontsize(8)
    at.set_fontweight('bold')
    at.set_color('white')
for t in texts:
    t.set_fontsize(9)
# Sin titulo principal (segun solicitud)
plt.tight_layout()
save_fig(fig, '06_torta_recorte_gasto')
plt.show()

# ── Grafico 07: Barras con leyenda manual (baja/suba x2 anos) ─────────
df_ir_s = df_ir.sort_values(COL_V2)
y_pos   = np.arange(len(df_ir_s))
h       = 0.35
v1_vals = df_ir_s[COL_V1].fillna(0)
v2_vals = df_ir_s[COL_V2].fillna(0)
c24 = ['#c0392b' if v < 0 else '#1a5276' for v in v1_vals]
c25 = ['#922b21' if v < 0 else '#154360' for v in v2_vals]

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(y_pos + h/2, v1_vals, height=h, color=c24,
        alpha=0.85, edgecolor='white', linewidth=0.4)
ax.barh(y_pos - h/2, v2_vals, height=h, color=c25,
        alpha=0.85, edgecolor='white', linewidth=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_ir_s['Rubro'], fontsize=9)
ax.axvline(0, color='#2c3e50', lw=1.2)
ax.set_xlabel(f'Variacion real anual | Billones ARS ({BASE.strftime("%b %Y")})')
ax.grid(axis='x', alpha=0.2)
ax.spines['left'].set_visible(False)
ax.set_title(
    f'Variacion del gasto por rubro: {ANO_BASE} vs {ANO_1} y {ANO_2}',
    fontsize=11, pad=10
)
# Leyenda manual: 4 colores (baja/suba para cada año), ubicada abajo a la derecha
leg = [
    Patch(facecolor='#c0392b', alpha=0.85, label=f'{ANO_BASE}->{ANO_1} (baja)'),
    Patch(facecolor='#1a5276', alpha=0.85, label=f'{ANO_BASE}->{ANO_1} (suba)'),
    Patch(facecolor='#922b21', alpha=0.85, label=f'{ANO_BASE}->{ANO_2} (baja)'),
    Patch(facecolor='#154360', alpha=0.85, label=f'{ANO_BASE}->{ANO_2} (suba)'),
]
ax.legend(handles=leg, fontsize=8, loc='lower right',
          bbox_to_anchor=(1.0, 0.0), framealpha=0.92, ncol=2)
plt.tight_layout()
save_fig(fig, '07_recorte_por_rubro')
plt.show()

In [ ]:
# ── Celda 9: Resumen completo del ajuste fiscal ───────────────────────
PIB_B = {2020:44.9, 2021:72.0, 2022:115.0, 2023:143.2, 2024:586.7, 2025:725.0}

print('=' * 80)
print('RESUMEN DEL AJUSTE FISCAL - GESTION MILEI')
print(f'Comparacion: anos completos {ANO_BASE} vs {ANO_1} y {ANO_2}')
print(f'Pesos constantes de {BASE.strftime("%b-%Y")} | B = billones = 10^12 pesos')
print('=' * 80)

df_macro = pd.DataFrame()
for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',               'Prim_real'),
    ('XV_RESULTADO_FINANCIERO',              'Fin_real'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto_real'),
    ('I_INGRESOS_CORRIENTES',                'Ing_real'),
    ('II2_INTERESES',                        'Int_real'),
]:
    df_macro[col] = get_real_anual_total(concepto)

for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',  'Prim_nom'),
    ('XV_RESULTADO_FINANCIERO', 'Fin_nom'),
]:
    s = get_serie_total(concepto)
    df_macro[col] = s.groupby(s.index.year).sum() / 1e6

df_macro['Prim_pct_PIB'] = df_macro.apply(
    lambda r: f"{r['Prim_nom']/PIB_B[r.name]*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)
df_macro['Fin_pct_PIB']  = df_macro.apply(
    lambda r: f"{r['Fin_nom']/PIB_B[r.name]*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)

print()
print('1. RESULTADO FISCAL ANUAL - SECTOR PUBLICO TOTAL')
t1 = df_macro[['Prim_real','Fin_real','Prim_pct_PIB','Fin_pct_PIB','Ing_real','Gasto_real','Int_real']]
t1 = t1.rename(columns={'Prim_real':'Primario(B)','Fin_real':'Financiero(B)',
                          'Prim_pct_PIB':'Prim%PIB','Fin_pct_PIB':'Fin%PIB',
                          'Ing_real':'Ingresos(B)','Gasto_real':'Gasto prim.(B)',
                          'Int_real':'Intereses(B)'})
print(t1[t1.index.isin(range(2020, 2027))].round(1).to_string())
print('  * PIB aproximado | Fuentes: INDEC, Hacienda')

gp23 = df_macro.loc[ANO_BASE,'Gasto_real']
gp24 = df_macro.loc[ANO_1,   'Gasto_real']
gp25 = df_macro.loc[ANO_2,   'Gasto_real']
rp23 = df_macro.loc[ANO_BASE,'Prim_real']
rp24 = df_macro.loc[ANO_1,   'Prim_real']
rp25 = df_macro.loc[ANO_2,   'Prim_real']
rf23 = df_macro.loc[ANO_BASE,'Fin_real']
rf24 = df_macro.loc[ANO_1,   'Fin_real']
rf25 = df_macro.loc[ANO_2,   'Fin_real']

print()
print(f'2. MAGNITUD DEL AJUSTE ({ANO_BASE} vs {ANO_1}/{ANO_2})')
print(f'   Gasto prim. {ANO_BASE}: {gp23:.1f} B | {ANO_1}: {gp24:.1f} B ({(gp24-gp23)/gp23*100:+.1f}%) | {ANO_2}: {gp25:.1f} B ({(gp25-gp23)/gp23*100:+.1f}%)')
print(f'   Result. primario  {ANO_BASE}: {rp23:+.1f} B | {ANO_1}: {rp24:+.1f} B | {ANO_2}: {rp25:+.1f} B')
print(f'   Result. financiero {ANO_BASE}: {rf23:+.1f} B | {ANO_1}: {rf24:+.1f} B | {ANO_2}: {rf25:+.1f} B')
print(f'   Mejora primaria {ANO_BASE}->{ANO_1}: {rp24-rp23:+.1f} B | {ANO_BASE}->{ANO_2}: {rp25-rp23:+.1f} B')

print()
print(f'3. DESGLOSE POR COMPONENTE AIF (Adm. Nacional, {ANO_BASE} vs {ANO_1}/{ANO_2})')
print(df_aif_comp.to_string(index=False))

print()
print(f'4. DESGLOSE FUNCIONAL POR RUBRO (IMIG, {ANO_BASE} vs {ANO_2})')
print(df_ir.to_string(index=False))
baja_t = df_ir[df_ir[COL_V2] < 0][COL_V2].sum()
suba_t = df_ir[df_ir[COL_V2] > 0][COL_V2].sum()
print(f'   Total baja: {baja_t:.1f} B | Total suba: {suba_t:+.1f} B | Neto: {baja_t+suba_t:.1f} B')
print()
print('   Notas: comparacion anos completos | AIF=base caja | IMIG=consolidado')
print('   2026 parcial no incluido | PIB aproximado, solo para orden de magnitud')

# ── Tablas "listas para informe" (se exportan en la celda 10) ─────────
# Reutilizan df_macro (Sector Publico Total, real, base ultimo mes) ya construido arriba.
_anios_inf = [a for a in range(2020, 2027) if a in df_macro.index]

df_informe_macro = pd.DataFrame({
    'anio':                   _anios_inf,
    'ingresos_corrientes_B':  [round(df_macro.loc[a, 'Ing_real'],   1) for a in _anios_inf],
    'gasto_primario_B':       [round(df_macro.loc[a, 'Gasto_real'], 1) for a in _anios_inf],
    'intereses_netos_B':      [round(df_macro.loc[a, 'Int_real'],   1) for a in _anios_inf],
    'resultado_primario_B':   [round(df_macro.loc[a, 'Prim_real'],  1) for a in _anios_inf],
    'resultado_financiero_B': [round(df_macro.loc[a, 'Fin_real'],   1) for a in _anios_inf],
    'primario_pct_PIB':       [df_macro.loc[a, 'Prim_pct_PIB'] for a in _anios_inf],
    'financiero_pct_PIB':     [df_macro.loc[a, 'Fin_pct_PIB']  for a in _anios_inf],
})

# Transferencias a provincias: subsector total_adm_nacional, real, en billones (1 decimal)
_corr_inf = get_real_anual('II4b1_TRANSF_PROVINCIAS_CABA', 'total_adm_nacional')
_cap_inf  = get_real_anual('V2a_TRANSF_CAPITAL_PROVINCIAS', 'total_adm_nacional')
_prov_rows = []
for a in _anios_inf:
    _cu = _corr_inf.get(a, 0.0)
    _ku = _cap_inf.get(a, 0.0)
    corr = round(_cu, 1)
    cap  = round(_ku, 1)
    tot  = round(_cu + _ku, 1)   # suma desde NO redondeados (regla informe)
    gp   = df_macro.loc[a, 'Gasto_real'] if a in df_macro.index else None
    pct  = round((_cu + _ku) / gp * 100, 1) if gp else None
    _prov_rows.append((a, corr, cap, tot, pct))
df_informe_prov = pd.DataFrame(_prov_rows,
    columns=['anio', 'corrientes_B', 'capital_B', 'total_B', 'pct_gasto_primario'])

print()
print('Tablas para informe listas: df_informe_macro, df_informe_prov')


# ── Valores y tablas LaTeX listos para el informe (export en celda 10) ─
# Regla: toda variacion/porcentaje se calcula desde valores reales SIN redondear
# y se redondea solo al formatear (mismo criterio que Informe_provincias corregido).
_MESES_FULL = {1:'enero',2:'febrero',3:'marzo',4:'abril',5:'mayo',6:'junio',
               7:'julio',8:'agosto',9:'septiembre',10:'octubre',11:'noviembre',12:'diciembre'}
BASE_TXT = f'{_MESES_FULL[BASE.month]} de {BASE.year}'
_idx_cov = get_serie_total('XIV_RESULTADO_PRIMARIO').index
_primer  = _idx_cov.min()
COBERTURA_TXT = f'{_MESES_FULL[_primer.month]} de {_primer.year} a {BASE_TXT}'

MENOS = '−'  # signo menos tipografico, como en el informe
def _c(s): return s.replace('.', ',')
# Formatos para la hoja Informe_valores (texto legible para transcribir)
def fmag(x, d=1):  return _c(f'{round(float(x), d):.{d}f}')
def fsig(x, d=1):
    v = round(float(x), d); return ('+' if v >= 0 else MENOS) + _c(f'{abs(v):.{d}f}')
def fpcts(x, d=1):
    v = round(float(x), d); return ('+' if v >= 0 else MENOS) + _c(f'{abs(v):.{d}f}') + r'\%'
def fpctp(x, d=1): return _c(f'{round(float(x), d):.{d}f}') + r'\%'

# Accesos a series reales anuales (sin redondear)
Ing  = df_macro['Ing_real'];  Prim = df_macro['Prim_real']; Fin = df_macro['Fin_real']
GP   = df_macro['Gasto_real']; Intn = df_macro['Int_real']
def ANr(cod):   return get_real_anual(cod, 'total_adm_nacional')
def TOTr(cod):  return get_real_anual_total(cod)
def IMr(cod, n):return get_imig_anual(cod, n)
def ppib(col, y): return df_macro.loc[y, col] / PIB_B[y] * 100

# Transferencias a provincias (total_adm_nacional, sin redondear) por anio
_tt = {y: _corr_inf.get(y, 0.0) + _cap_inf.get(y, 0.0) for y in df_macro.index}
def pctGP(y): return _tt[y] / GP[y] * 100

_F = []
def _add(sec, clave, desc, num, fmt, uni): _F.append((sec, clave, desc, num, fmt, uni))

# ---- Portada / Titulo ----
_v = GP[2024] - GP[2023]
_add('Portada', 'gp_ajuste_2324_B', 'Ajuste gasto primario 2023->2024', round(_v,1), fsig(_v), 'B')
_p = (GP[2024]/GP[2023]-1)*100
_add('Portada', 'gp_ajuste_2324_pct', 'Ajuste gasto primario 2023->2024 (%)', round(_p,1), fpcts(_p), '%')
_pp = ppib('Prim_nom',2024) - ppib('Prim_nom',2023)
_add('Portada', 'mejora_pp_PIB', 'Mejora resultado primario en puntos del PIB', round(_pp,1), fmag(_pp), 'pp_PIB')
_vt = _tt[2024] - _tt[2023]
_add('Portada', 'transf_caida_2324_B', 'Caida transferencias a provincias 2023->2024', round(_vt,1), fsig(_vt), 'B')
_sh = _vt / _v * 100
_add('Portada', 'transf_pct_del_ajuste', 'Transferencias como % del ajuste del gasto primario', round(_sh,1), fpctp(_sh), '%')
_pt = (_tt[2024]/_tt[2023]-1)*100
_add('Portada', 'transf_pct_real_2324', 'Caida real transferencias 2023->2024 (%)', round(_pt,0), fpcts(_pt,0), '%')
# obra publica (IMIG capital) para portada
_ob = IMr('Gastos_capital',1)
_obv = _ob[2025]-_ob[2023]; _obp = (_ob[2025]/_ob[2023]-1)*100
_add('Portada', 'obra_monto_2325_B', 'Caida obra publica 2023->2025 (IMIG)', round(_obv,1), fsig(_obv), 'B')
_add('Portada', 'obra_pct_real', 'Caida real obra publica 2023->2025 (%)', round(_obp,0), fpcts(_obp,0), '%')
_add('Portada', 'mes_base', 'Mes base del deflactor', None, BASE_TXT, 'texto')
_add('Portada', 'cobertura', 'Cobertura temporal de los datos', None, COBERTURA_TXT, 'texto')

# ---- Resumen ejecutivo / Evolucion ----
_sp = get_serie_total('XIV_RESULTADO_PRIMARIO')
_sp1 = _sp[_sp.index.year <= 2023]; _sp2 = _sp[_sp.index.year >= 2024]
_add('Resumen', 'meses_superavit_2023', 'Meses con superavit primario 2020-2023', int((_sp1>=0).sum()), str(int((_sp1>=0).sum())), 'meses')
_add('Resumen', 'meses_total_2023', 'Total meses 2020-2023', int(len(_sp1)), str(int(len(_sp1))), 'meses')
_add('Resumen', 'meses_deficit_desde2024', 'Meses con deficit primario desde ene-2024', int((_sp2<0).sum()), str(int((_sp2<0).sum())), 'meses')
_add('Resumen', 'meses_total_desde2024', 'Total meses desde ene-2024', int(len(_sp2)), str(int(len(_sp2))), 'meses')
_mp = Prim[2024]-Prim[2023]
_add('Resumen', 'mejora_primario_2324_B', 'Mejora resultado primario 2023->2024', round(_mp,1), fsig(_mp), 'B')
_add('Resumen', 'mejora_primario_pp_PIB', 'Mejora resultado primario en puntos del PIB', round(_pp,1), fmag(_pp), 'pp_PIB')
_add('Resumen', 'primario_2023_B', 'Resultado primario 2023', round(Prim[2023],1), fsig(Prim[2023]), 'B')
_add('Resumen', 'primario_2023_pct_PIB', 'Resultado primario 2023 (% PIB)', round(ppib('Prim_nom',2023),1), fpcts(ppib('Prim_nom',2023)), '%')
_add('Resumen', 'primario_2024_B', 'Resultado primario 2024', round(Prim[2024],1), fsig(Prim[2024]), 'B')
_add('Resumen', 'primario_2024_pct_PIB', 'Resultado primario 2024 (% PIB)', round(ppib('Prim_nom',2024),1), fpcts(ppib('Prim_nom',2024)), '%')
_add('Resumen', 'gasto_primario_2023_B', 'Gasto primario 2023', round(GP[2023],1), fmag(GP[2023]), 'B')
_add('Resumen', 'gasto_primario_2024_B', 'Gasto primario 2024', round(GP[2024],1), fmag(GP[2024]), 'B')
_add('Resumen', 'gasto_primario_2025_B', 'Gasto primario 2025', round(GP[2025],1), fmag(GP[2025]), 'B')
_add('Resumen', 'ingresos_2023_B', 'Ingresos corrientes 2023', round(Ing[2023],1), fmag(Ing[2023]), 'B')
_add('Resumen', 'ingresos_2024_B', 'Ingresos corrientes 2024', round(Ing[2024],1), fmag(Ing[2024]), 'B')
_pi24 = (Ing[2024]/Ing[2023]-1)*100
_add('Resumen', 'ingresos_2024_var', 'Ingresos corrientes var 2023->2024', round(_pi24,1), fpcts(_pi24), '%')
_add('Resumen', 'ingresos_2025_B', 'Ingresos corrientes 2025', round(Ing[2025],1), fmag(Ing[2025]), 'B')
_pi25 = (Ing[2025]/Ing[2023]-1)*100
_add('Resumen', 'ingresos_2025_var_acum', 'Ingresos corrientes var acumulada 2023->2025', round(_pi25,1), fpcts(_pi25), '%')
_add('Resumen', 'ingresos_2026_B', 'Ingresos corrientes 2026 (5 meses)', round(Ing[2026],1), fmag(Ing[2026]), 'B')
_ci = Ing[2024]-Ing[2023]
_add('Resumen', 'ingresos_caida_2324_B', 'Caida ingresos corrientes 2023->2024', round(_ci,1), fsig(_ci), 'B')
_add('Resumen', 'primario_2025_B', 'Superavit primario 2025', round(Prim[2025],1), fsig(Prim[2025]), 'B')
_add('Resumen', 'primario_2025_pct_PIB', 'Resultado primario 2025 (% PIB)', round(ppib('Prim_nom',2025),1), fpcts(ppib('Prim_nom',2025)), '%')
_add('Resumen', 'financiero_2023_B', 'Resultado financiero 2023', round(Fin[2023],1), fsig(Fin[2023]), 'B')
_add('Resumen', 'financiero_2024_B', 'Resultado financiero 2024', round(Fin[2024],1), fsig(Fin[2024]), 'B')
_add('Resumen', 'financiero_2025_B', 'Resultado financiero 2025', round(Fin[2025],1), fsig(Fin[2025]), 'B')

# ---- Composicion del gasto ----
_GCt = TOTr('II_GASTOS_CORRIENTES')
_add('Composicion', 'gasto_corriente_max_2020_B', 'Gasto corriente maximo 2020 (Sector Publico Total)', round(_GCt[2020],1), fmag(_GCt[2020]), 'B')
_add('Composicion', 'gasto_corriente_2023_B', 'Gasto corriente 2023', round(_GCt[2023],1), fmag(_GCt[2023]), 'B')
_add('Composicion', 'gasto_corriente_2024_B', 'Gasto corriente 2024', round(_GCt[2024],1), fmag(_GCt[2024]), 'B')
_gc24 = (_GCt[2024]/_GCt[2023]-1)*100
_add('Composicion', 'gasto_corriente_2024_var', 'Gasto corriente var 2023->2024', round(_gc24,1), fpcts(_gc24), '%')
_add('Composicion', 'gasto_corriente_2025_B', 'Gasto corriente 2025', round(_GCt[2025],1), fmag(_GCt[2025]), 'B')
_gc25 = (_GCt[2025]/_GCt[2023]-1)*100
_add('Composicion', 'gasto_corriente_2025_var', 'Gasto corriente var 2023->2025', round(_gc25,1), fpcts(_gc25), '%')
_pr = ANr('II3_PRESTACIONES_SEG_SOCIAL'); _prv = (_pr[2024]/_pr[2023]-1)*100
_add('Composicion', 'prestaciones_2024_var', 'Prestaciones Seg.Social var 2023->2024 (Adm.Nac)', round(_prv,1), fpcts(_prv), '%')
_su = ANr('II4a_TRANSF_SECTOR_PRIVADO')
_su24 = (_su[2024]/_su[2023]-1)*100; _su25 = (_su[2025]/_su[2023]-1)*100
_add('Composicion', 'subsidios_2024_var', 'Subsidios economicos var 2023->2024', round(_su24,1), fpcts(_su24), '%')
_add('Composicion', 'subsidios_2025_var', 'Subsidios economicos var 2023->2025', round(_su25,1), fpcts(_su25), '%')
_suv = _su[2025]-_su[2023]
_add('Composicion', 'subsidios_caida_2325_B', 'Subsidios economicos caida acumulada 2023->2025', round(_suv,1), fsig(_suv), 'B')
_tp = ANr('II4b1_TRANSF_PROVINCIAS_CABA'); _tpv = (_tp[2024]/_tp[2023]-1)*100
_add('Composicion', 'transf_prov_2024_var', 'Transf. Provincias/CABA var 2023->2024 (Adm.Nac)', round(_tpv,1), fpcts(_tpv), '%')
_re = ANr('II1a_REMUNERACIONES')
_re24 = (_re[2024]/_re[2023]-1)*100; _re25 = (_re[2025]/_re[2023]-1)*100
_add('Composicion', 'remuneraciones_2024_var', 'Remuneraciones var 2023->2024', round(_re24,1), fpcts(_re24), '%')
_add('Composicion', 'remuneraciones_2025_var', 'Remuneraciones var 2023->2025', round(_re25,1), fpcts(_re25), '%')
_ka = ANr('V_GASTOS_CAPITAL')
_ka24 = (_ka[2024]/_ka[2023]-1)*100; _ka25 = (_ka[2025]/_ka[2023]-1)*100
_add('Composicion', 'obra_aif_2024_var', 'Obra publica (capital AIF) var 2023->2024', round(_ka24,1), fpcts(_ka24), '%')
_add('Composicion', 'obra_aif_2023_B', 'Gastos de capital 2023 (AIF Adm.Nac)', round(_ka[2023],1), fmag(_ka[2023]), 'B')
_add('Composicion', 'obra_aif_2024_B', 'Gastos de capital 2024 (AIF Adm.Nac)', round(_ka[2024],1), fmag(_ka[2024]), 'B')
_add('Composicion', 'obra_aif_2025_B', 'Gastos de capital 2025 (AIF Adm.Nac)', round(_ka[2025],1), fmag(_ka[2025]), 'B')
_add('Composicion', 'obra_aif_2025_var', 'Obra publica (capital AIF) var 2023->2025', round(_ka25,1), fpcts(_ka25), '%')
for _y, _k in [(2021,'ingresos_pico_2021_B'),(2022,'ingresos_2022_B'),(2023,'ingresos_2023b_B'),(2024,'ingresos_2024b_B'),(2025,'ingresos_2025b_B')]:
    _add('Composicion', _k, f'Ingresos corrientes {_y} (Sector Publico Total)', round(Ing[_y],1), fmag(Ing[_y]), 'B')

# ---- Transferencias a provincias ----
_pcts_hist = [pctGP(y) for y in range(2020,2024)]
_add('Transferencias', 'pctGP_hist_min', 'Minimo %GP transferencias 2020-2023', round(min(_pcts_hist),1), fpctp(min(_pcts_hist)), '%')
_add('Transferencias', 'pctGP_hist_max', 'Maximo %GP transferencias 2020-2023', round(max(_pcts_hist),1), fpctp(max(_pcts_hist)), '%')
_add('Transferencias', 'pctGP_2024', '%GP transferencias 2024', round(pctGP(2024),1), fpctp(pctGP(2024)), '%')
_add('Transferencias', 'pctGP_2025', '%GP transferencias 2025', round(pctGP(2025),1), fpctp(pctGP(2025)), '%')
_cap = ANr('V2a_TRANSF_CAPITAL_PROVINCIAS'); _capv = (_cap[2024]/_cap[2023]-1)*100
_add('Transferencias', 'capital_2324_var', 'Capital var 2023->2024 (Adm.Nac)', round(_capv,1), fpcts(_capv), '%')
_add('Transferencias', 'capital_2023_B', 'Transf. capital 2023', round(_cap[2023],1), fmag(_cap[2023]), 'B')
_add('Transferencias', 'capital_2024_B', 'Transf. capital 2024', round(_cap[2024],1), fmag(_cap[2024]), 'B')
_corr = ANr('II4b1_TRANSF_PROVINCIAS_CABA'); _corrv = (_corr[2024]/_corr[2023]-1)*100
_add('Transferencias', 'corrientes_2324_var', 'Corrientes var 2023->2024 (Adm.Nac)', round(_corrv,1), fpcts(_corrv), '%')
_add('Transferencias', 'corrientes_2023_B', 'Transf. corrientes 2023', round(_corr[2023],1), fmag(_corr[2023]), 'B')
_add('Transferencias', 'corrientes_2024_B', 'Transf. corrientes 2024', round(_corr[2024],1), fmag(_corr[2024]), 'B')
_add('Transferencias', 'total_2025_B', 'Transferencias total 2025', round(_tt[2025],1), fmag(_tt[2025]), 'B')
_dt = (_tt[2025]/_tt[2023]-1)*100
_add('Transferencias', 'total_2025_dist_2023', 'Distancia transferencias 2025 vs 2023', round(_dt,0), fpcts(_dt,0), '%')
_add('Transferencias', 'pctGP_2023b', '%GP transferencias 2023', round(pctGP(2023),1), fpctp(pctGP(2023)), '%')
_add('Transferencias', 'pctGP_2025b', '%GP transferencias 2025', round(pctGP(2025),1), fpctp(pctGP(2025)), '%')

# ---- Recorte por rubro (IMIG), 2023->2025 ----
_imig_pros = [
    ('obra_publica','Obra publica','Gastos_capital',1),
    ('subsidios','Subsidios economicos','Subsidios_economicos',1),
    ('otros_prog','Otros programas sociales','Otros_prog_sociales',2),
    ('salarios','Salarios estatales','Salarios',2),
    ('transf_prov','Transf. corrientes a provincias','Transf_corrientes_provincias',1),
    ('universidades','Universidades nacionales','Transf_universidades',1),
    ('pami','PAMI (INSSJP)','INSSJP_PAMI',2),
    ('jubilaciones','Jubilaciones contributivas','Jubilaciones_pensiones',2),
    ('auh','AUH','AUH',2),
]
for _k,_lbl,_cod,_n in _imig_pros:
    _a = IMr(_cod,_n); _vv = _a[2025]-_a[2023]; _vp = (_a[2025]/_a[2023]-1)*100
    _add('Recorte_IMIG', f'{_k}_2325_B', f'{_lbl}: var 2023->2025', round(_vv,1), fsig(_vv), 'B')
    _add('Recorte_IMIG', f'{_k}_2325_pct', f'{_lbl}: var % 2023->2025', round(_vp,1), fpcts(_vp), '%')
# AUH niveles + jubilaciones caida 2024
_auh = IMr('AUH',2)
_add('Recorte_IMIG', 'auh_2023_B', 'AUH 2023', round(_auh[2023],1), fmag(_auh[2023]), 'B')
_add('Recorte_IMIG', 'auh_2025_B', 'AUH 2025', round(_auh[2025],1), fmag(_auh[2025]), 'B')
_ju = IMr('Jubilaciones_pensiones',2); _ju24 = (_ju[2024]/_ju[2023]-1)*100
_add('Recorte_IMIG', 'jubilaciones_2024_var', 'Jubilaciones caida 2024 vs 2023', round(_ju24,1), fpcts(_ju24), '%')
# Participaciones en la torta (mismas vars que la figura 06: df_ir[COL_V2], ya redondeadas)
_baja = df_ir[df_ir[COL_V2] < 0].copy()
_baja_tot = _baja[COL_V2].abs().sum()
_torta_map = {'Obra publica':'obra_publica','Subsidios':'subsidios','Otros prog. sociales':'otros_prog',
              'Salarios':'salarios','Transf. Provincias':'transf_prov','Universidades':'universidades',
              'Pensiones NC':'pensiones_nc'}
for _, _row in _baja.iterrows():
    _rb = _row['Rubro']; _share = abs(_row[COL_V2]) / _baja_tot * 100
    _add('Recorte_IMIG', f'torta_{_torta_map.get(_rb, _rb)}', f'Participacion en la baja: {_rb}', round(_share,1), fpctp(_share), '%')

# ---- Conclusiones ----
_add('Conclusiones', 'gp_var_2024', 'Gasto primario var 2024 vs 2023', round(_p,1), fpcts(_p), '%')
_add('Conclusiones', 'deficit_2023_mag_B', 'Deficit primario 2023 (magnitud)', round(abs(Prim[2023]),1), fmag(abs(Prim[2023])), 'B')
_add('Conclusiones', 'superavit_2024_mag_B', 'Superavit primario 2024 (magnitud)', round(abs(Prim[2024]),1), fmag(abs(Prim[2024])), 'B')
_o = IMr('Gastos_capital',1); _add('Conclusiones', 'obra_pct_concl', 'Obra publica var 2023->2025 (%)', round((_o[2025]/_o[2023]-1)*100,0), fpcts((_o[2025]/_o[2023]-1)*100,0), '%')
_s = IMr('Subsidios_economicos',1); _add('Conclusiones', 'subsidios_pct_concl', 'Subsidios var 2023->2025 (%)', round((_s[2025]/_s[2023]-1)*100,0), fpcts((_s[2025]/_s[2023]-1)*100,0), '%')
_tpi = IMr('Transf_corrientes_provincias',1); _add('Conclusiones', 'transf_pct_concl', 'Transf. provincias var 2023->2025 (%)', round((_tpi[2025]/_tpi[2023]-1)*100,0), fpcts((_tpi[2025]/_tpi[2023]-1)*100,0), '%')
_ah = IMr('AUH',2); _add('Conclusiones', 'auh_pct_concl', 'AUH var 2023->2025 (%)', round((_ah[2025]/_ah[2023]-1)*100,0), fpcts((_ah[2025]/_ah[2023]-1)*100,0), '%')

# ---- Metodologia (insumos fijos de Hacienda, NO recalcular) ----
_add('Metodologia', 'pib_resultado_2023', 'Resultado financiero 2023 oficial (% PIB, Hacienda)', -6.1, MENOS+'6,1'+r'\%', '%')
_add('Metodologia', 'pib_resultado_2024', 'Resultado financiero 2024 oficial (% PIB, Hacienda)', 0.3, '+0,3'+r'\%', '%')
_add('Metodologia', 'pib_resultado_2025', 'Resultado financiero 2025 oficial (% PIB, Hacienda)', 0.2, '+0,2'+r'\%', '%')

_orden_sec = ['Portada','Resumen','Composicion','Transferencias','Recorte_IMIG','Conclusiones','Metodologia']
df_informe_valores = pd.DataFrame(_F, columns=['seccion','clave','descripcion','valor_num','valor_fmt','unidad'])
df_informe_valores['seccion'] = pd.Categorical(df_informe_valores['seccion'], categories=_orden_sec, ordered=True)
df_informe_valores = df_informe_valores.sort_values('seccion', kind='stable').reset_index(drop=True)

# ── Bloques LaTeX de las 5 tablas (solo filas de datos entre encabezado y \bottomrule) ──
def lxn(x, d=1):  return _c(f'{round(float(x), d):.{d}f}')
def lxd(x, d=1):
    v = round(float(x), d); return (r'$+$' if v >= 0 else r'$-$') + _c(f'{abs(v):.{d}f}')
def lxdp(x, d=1):
    v = round(float(x), d); return (r'$+$' if v >= 0 else r'$-$') + _c(f'{abs(v):.{d}f}') + r'\,\%'
def lxbold(x, d=1):
    v = round(float(x), d); s = r'$\mathbf{-}$' if v < 0 else r'$\mathbf{+}$'
    return s + r'\textbf{' + _c(f'{abs(v):.{d}f}') + '}'
def lxpp(x, d=1):
    v = round(float(x), d); return ('+' if v >= 0 else '-') + _c(f'{abs(v):.{d}f}')
def lxppp(x, d=1):
    v = round(float(x), d); return ('+' if v >= 0 else '-') + _c(f'{abs(v):.{d}f}') + r'\,\%'
def lxplain_pct(x, d=1): return _c(f'{round(float(x), d):.{d}f}') + r'\,\%'

_y7 = [2020,2021,2022,2023,2024,2025,2026]
def _pibrow(col):
    return [lxdp(df_macro.loc[y, col]/PIB_B[y]*100) if y in PIB_B else '---' for y in _y7]

# tab:01
_L = [r'\multicolumn{7}{@{}l}{\textit{Billones de pesos constantes de ' + BASE_TXT + r'}} \\[2pt]']
_L.append('Ingresos corrientes & ' + ' & '.join(lxn(Ing[y]) for y in _y7) + r' \\')
_L.append('Gasto primario & ' + ' & '.join(lxn(GP[y]) for y in _y7) + r' \\')
_L.append('Intereses netos & ' + ' & '.join(lxn(Intn[y]) for y in _y7) + r' \\[4pt]')
_L.append(r'\midrule')
_L.append('Resultado primario & ' + ' & '.join(lxd(Prim[y]) for y in _y7) + r' \\')
_L.append(r'\quad \% del PIB & ' + ' & '.join(_pibrow('Prim_nom')) + r' \\[2pt]')
_L.append('Resultado financiero & ' + ' & '.join(lxd(Fin[y]) for y in _y7) + r' \\')
_L.append(r'\quad \% del PIB & ' + ' & '.join(_pibrow('Fin_nom')) + r' \\')
tex_01 = '\n'.join(_L)

# tab:composicion_gasto (Adm. Nacional)
_comp = [('Prestaciones Seg. Social','II3_PRESTACIONES_SEG_SOCIAL'),
         ('Subsidios económicos','II4a_TRANSF_SECTOR_PRIVADO'),
         ('Remuneraciones','II1a_REMUNERACIONES'),
         ('Transf. Provincias/CABA','II4b1_TRANSF_PROVINCIAS_CABA'),
         ('Universidades','II4b2_TRANSF_UNIVERSIDADES'),
         ('Intereses','II2_INTERESES')]
_L = [r'\multicolumn{5}{@{}l}{\textit{Billones de pesos constantes de ' + BASE_TXT + r'}} \\[2pt]']
for _lbl, _cod in _comp:
    _a = ANr(_cod); _v23, _v24, _v25 = _a[2023], _a[2024], _a[2025]
    _L.append(f'{_lbl} & ' + lxn(_v23) + ' & ' + lxn(_v24) + ' & ' + lxn(_v25) + ' & ' + lxd(_v24-_v23) + ' & ' + lxd(_v25-_v23) + r' \\')
_L.append(r'\midrule')
_gc = ANr('II_GASTOS_CORRIENTES'); _g23, _g24, _g25 = _gc[2023], _gc[2024], _gc[2025]
_L.append(r'\textbf{Total gasto corriente} & \textbf{' + lxn(_g23) + r'} & \textbf{' + lxn(_g24) + r'} & \textbf{' + lxn(_g25) + r'} & ' + lxbold(_g24-_g23) + ' & ' + lxbold(_g25-_g23) + r' \\')
tex_comp = '\n'.join(_L)

# tab:ing_gasto_anual (Sector Publico Total)
_GCt2 = TOTr('II_GASTOS_CORRIENTES'); _Prest = TOTr('II3_PRESTACIONES_SEG_SOCIAL')
_L = [r'\multicolumn{7}{@{}l}{\textit{Billones de pesos constantes de ' + BASE_TXT + r'}} \\[2pt]']
_L.append('Ingresos corrientes & ' + ' & '.join(lxn(Ing[y]) for y in _y7) + r' \\')
_L.append('Gasto corriente & ' + ' & '.join(lxn(_GCt2[y]) for y in _y7) + r' \\')
_L.append(r'\quad Intereses & ' + ' & '.join(lxn(Intn[y]) for y in _y7) + r' \\')
_L.append(r'\quad Prestaciones & ' + ' & '.join(lxn(_Prest[y]) for y in _y7) + r' \\')
_L.append(r'\midrule')
_L.append(r'\multicolumn{7}{@{}l}{\textit{Variación real anual}} \\[2pt]')
def _yoy(ser):
    out = []
    for y in _y7:
        if y in (2020, 2026) or (y-1) not in ser.index or y not in ser.index: out.append('---')
        else: out.append(lxdp((ser[y]/ser[y-1]-1)*100))
    return out
_L.append('Ingresos corrientes & ' + ' & '.join(_yoy(Ing)) + r' \\')
_L.append('Gasto corriente & ' + ' & '.join(_yoy(_GCt2)) + r' \\')
tex_ing = '\n'.join(_L)

# tab:prov (suma desde no redondeados)
_L = []
for y in range(2020, 2026):
    _cu = _corr_inf.get(y, 0.0); _ku = _cap_inf.get(y, 0.0); _to = _cu + _ku
    _L.append(f'{y} & ' + lxn(_cu) + ' & ' + lxn(_ku) + ' & ' + lxn(_to) + ' & ' + lxplain_pct(_to/GP[y]*100) + r' \\')
tex_prov = '\n'.join(_L)

# tab:imig (orden del documento)
_imig_doc = [('Jubilaciones contributivas','Jubilaciones_pensiones',2),
             ('Salarios estatales','Salarios',2),
             ('Subsidios económicos','Subsidios_economicos',1),
             ('Otros programas sociales','Otros_prog_sociales',2),
             ('PAMI (INSSJP)','INSSJP_PAMI',2),
             ('Obra pública (gastos de capital)','Gastos_capital',1),
             ('Transf. corrientes a provincias','Transf_corrientes_provincias',1),
             ('Universidades nacionales','Transf_universidades',1),
             ('Pensiones no contributivas','Pensiones_no_contributivas',2),
             ('AUH','AUH',2)]
_L = []
for _lbl, _cod, _n in _imig_doc:
    _a = IMr(_cod, _n); _v23, _v24, _v25 = _a[2023], _a[2024], _a[2025]
    _L.append(f'{_lbl} & ' + lxn(_v23) + ' & ' + lxn(_v24) + ' & ' + lxn(_v25) + ' & ' + lxpp(_v25-_v23) + ' & ' + lxppp((_v25/_v23-1)*100) + r' \\')
tex_imig = '\n'.join(_L)

TEX_FILES = {'tabla_01.tex': tex_01, 'tabla_composicion_gasto.tex': tex_comp,
             'tabla_ing_gasto_anual.tex': tex_ing, 'tabla_prov.tex': tex_prov,
             'tabla_imig.tex': tex_imig}
_TEX_LABELS = [('tabla_01.tex','tab:01_resultado_primario_financiero'),
               ('tabla_composicion_gasto.tex','tab:composicion_gasto'),
               ('tabla_ing_gasto_anual.tex','tab:ing_gasto_anual'),
               ('tabla_prov.tex','tab:prov'),
               ('tabla_imig.tex','tab:imig')]
_tl = []
for _fn, _lab in _TEX_LABELS:
    _tl.append('% === ' + _lab + ' ===')
    _tl.extend(TEX_FILES[_fn].split('\n'))
    _tl.append('')
df_tablas_latex = pd.DataFrame({'latex': _tl})

print()
print('Informe_valores:', len(df_informe_valores), 'filas | Tablas_LaTeX:', len(df_tablas_latex), 'lineas')


In [ ]:
# ── Celda 10: Exportar Excel + ZIP ───────────────────────────────────
EXCEL_FILE = 'analisis_fiscal_resultados.xlsx'
ZIP_FILE   = 'analisis_fiscal.zip'

idx_base = get_serie_total('XIV_RESULTADO_PRIMARIO').index
rows_m   = {'fecha': idx_base}
for nombre, cod in [
    ('ingresos_corrientes',  'I_INGRESOS_CORRIENTES'),
    ('gastos_corrientes',    'II_GASTOS_CORRIENTES'),
    ('intereses',            'II2_INTERESES'),
    ('prestaciones',         'II3_PRESTACIONES_SEG_SOCIAL'),
    ('resultado_primario',   'XIV_RESULTADO_PRIMARIO'),
    ('resultado_financiero', 'XV_RESULTADO_FINANCIERO'),
]:
    s = get_serie_total(cod)
    rows_m[f'{nombre}_nominal_MM'] = s.reindex(idx_base).values
    rows_m[f'{nombre}_real_MM']    = deflactar(s.reindex(idx_base))
df_serie = pd.DataFrame(rows_m)

df_anual_exp = df_serie.copy()
df_anual_exp['anio'] = pd.to_datetime(df_anual_exp['fecha']).dt.year
df_anual_exp = df_anual_exp.groupby('anio').sum(numeric_only=True)

prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA','Corrientes'),
                  ('V2a_TRANSF_CAPITAL_PROVINCIAS','Capital')]:
    for ss in ['tesoro_nacional','total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        prov_rows.append(pd.DataFrame({'fecha':s.index,'tipo':tipo,'subsector':ss,
                                        'nominal_MM':s.values,'real_MM':deflactar(s)}))
df_prov_exp = pd.concat(prov_rows).sort_values(['fecha','tipo'])

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
    df_serie.to_excel(writer,         sheet_name='Serie_mensual',       index=False)
    df_anual_exp.to_excel(writer,     sheet_name='Resumen_anual',       index=True)
    df_prov_exp.to_excel(writer,      sheet_name='Transferencias_prov', index=False)
    df_aif_comp.to_excel(writer,      sheet_name='Ajuste_AIF_anual',    index=False)
    df_ir.to_excel(writer,            sheet_name='Ajuste_IMIG_rubros',  index=False)
    df_informe_macro.to_excel(writer, sheet_name='Informe_tabla1',      index=False)
    df_informe_prov.to_excel(writer,  sheet_name='Informe_provincias',  index=False)
    df_informe_valores.to_excel(writer, sheet_name='Informe_valores',   index=False)
    df_tablas_latex.to_excel(writer,  sheet_name='Tablas_LaTeX',        index=False)

print(f'Excel: {EXCEL_FILE}')
print('Hojas: Serie_mensual | Resumen_anual | Transferencias_prov | Ajuste_AIF_anual | '
      'Ajuste_IMIG_rubros | Informe_tabla1 | Informe_provincias | Informe_valores | Tablas_LaTeX')

# Escribir los 5 .tex en disco (bloques de filas para pegar en el documento)
for _fn, _txt in TEX_FILES.items():
    with open(_fn, 'w', encoding='utf-8') as _f:
        _f.write(_txt + '\n')
print('Tablas LaTeX (.tex): ' + ', '.join(TEX_FILES))

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXCEL_FILE)
    for g in GRAFICOS:
        if os.path.exists(g): zf.write(g)
    for _fn in TEX_FILES:
        if os.path.exists(_fn): zf.write(_fn)

ngraf = len([g for g in GRAFICOS if os.path.exists(g)])
ntex  = len([f for f in TEX_FILES if os.path.exists(f)])
print(f'\nZIP: {ZIP_FILE} ({1+ngraf+ntex} archivos: Excel + {ngraf} graficos PNG + {ntex} .tex)')
for g in GRAFICOS:
    if os.path.exists(g):
        print(f'  {g}  ({os.path.getsize(g)//1024} KB)')

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)
    print('\nDescarga iniciada')
else:
    print(f'\nGuardado en: {os.path.abspath(ZIP_FILE)}')
